# 기존 Planner와 Runtime V2 비교

운영 중인 기존 RunPod 엔드포인트는 그대로 두고, 복제한 V2 테스트 엔드포인트와 같은 요청을 비교합니다. `.env`에 `RUNPOD_PLANNER_ENDPOINT_URL`, `RUNPOD_PLANNER_V2_ENDPOINT_URL`, `RUNPOD_API_KEY`를 설정한 뒤 커널을 재시작하고 실행하세요.

In [ ]:
import os, sys
from datetime import date, datetime
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')

from adapters.todo_creation.qwen_llm import DEFAULT_QWEN_MODEL
from adapters.todo_creation.runpod_llm import RunPodQwenLLM
from agents.todo_creation.planner.pipeline import PlannerPorts, run
from agents.todo_creation.schemas import PlannerInput

def build_ports(endpoint_url):
    if not endpoint_url:
        raise RuntimeError('기존/V2 RunPod 엔드포인트 URL을 모두 설정하세요.')
    common = dict(endpoint_url=endpoint_url, api_key=os.getenv('RUNPOD_API_KEY', 'EMPTY'), model=DEFAULT_QWEN_MODEL, max_tokens=1200)
    planner = RunPodQwenLLM(adapter='planner', **common)
    base = RunPodQwenLLM(adapter='base', **common)
    return PlannerPorts(llm=planner, classifier=base, validator=base)

PORTS = {
    '기존': build_ports(os.getenv('RUNPOD_PLANNER_ENDPOINT_URL', '').strip()),
    'V2': build_ports(os.getenv('RUNPOD_PLANNER_V2_ENDPOINT_URL', '').strip()),
}
TODAY = date.today()
print('ROOT:', ROOT)
print('TODAY:', TODAY)

In [ ]:
threads = {'기존': None, 'V2': None}

def reset(label=None):
    if label:
        threads[label] = None
    else:
        threads.update({'기존': None, 'V2': None})

async def send(label, message):
    result = await run(
        PlannerInput(user_id=f'ab-{label}', message=message, today=TODAY, thread_id=threads[label]),
        ports=PORTS[label], now=datetime.now(),
    )
    threads[label] = result.thread_id
    payload = result.model_dump(mode='json')
    print(f'\n[{label}] {message}')
    pprint(payload)
    return payload

async def compare_once(message):
    reset()
    old = await send('기존', message)
    new = await send('V2', message)
    return {'기존': old, 'V2': new}

## 같은 요청 한 번에 비교

정보가 충분한 요청으로 생성 결과의 관련성, 한국어 문장, 마지막 날짜를 비교합니다.

In [ ]:
# await compare_once('8월 8일 철인 삼종 경기에 출전하고 싶어. 입문자이고 주 4회 훈련 가능하며 안전하게 완주하는 게 목표야.')

In [ ]:
await compare_once('7월 25일까지 흑백요리사 지원용 대표 메뉴를 완성하고 싶어. 가정 요리 경험이 있고 주 4회 가능해.')

## 꼬리질문 대화 비교

각 모델은 별도 thread를 사용합니다. 첫 질문을 확인한 뒤 같은 정보를 각각 입력하세요.

In [ ]:
reset('기존')
await send('기존', '8월 8일 철인 삼종 경기에 출전하고 싶어')

In [ ]:
reset()
await send('기존', '슈퍼스타K에 출연하고 싶어')
await send('V2', '슈퍼스타K에 출연하고 싶어')

In [ ]:
await send('기존', '8월 말까지 지원 영상을 완성하고 싶고 노래 경험은 조금 있어')
await send('V2', '8월 말까지 지원 영상을 완성하고 싶고 노래 경험은 조금 있어')

In [ ]:
# await send('기존', '평일 저녁에 주 4회 가능하고 오디션 무대를 끝까지 해내는 게 목표야')
await send('V2', '평일 저녁에 주 4회 가능하고 오디션 무대를 끝까지 해내는 게 목표야')

In [ ]:
reset()
await send('V2', '안녕')

In [ ]:
await send('V2', '흑백요리사 우승하고 싶은데 어떻게 연습을 하는게 좋을까?')

In [ ]:
reset('V2')
await send('V2', '마라톤 완주하고싶은데 어떻게 연습을 하는게 좋을까?')

In [ ]:
reset('V2')
await send('V2', '흑백요리사 우승하고 싶은데 어떻게 연습하는 게 좋을까?')